# ETL dengue spatial cases

## 1. Load raw dataset

In [0]:
CATALOG = "workspace"
SCHEMA = "default"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

RAW_TABLE = f"{TABLE_PREFIX}.raw_dengue_espacial"
OUTPUT_TABLE = f"{TABLE_PREFIX}.fato_dengue_espacial"

df_raw = spark.table(RAW_TABLE)

display(df_raw)
print(df_raw.columns)

## 2. Define helper functions

In [0]:
from pyspark.sql.functions import (
    col, lit, trim, upper, regexp_replace, translate, to_date, when
)

ACCENTED = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑáàâãäéèêëíìîïóòôõöúùûüçñ"
UNACCENTED = "AAAAAEEEEIIIIOOOOOUUUUCNaaaaaeeeeiiiiooooouuuucn"

def normalize_text(column):
    return regexp_replace(
        upper(translate(trim(column.cast("string")), ACCENTED, UNACCENTED)),
        r"\s+",
        " "
    )

def clean_int(column_name):
    cleaned = regexp_replace(trim(col(column_name).cast("string")), r"[^0-9-]", "")
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("int"))

## 3. Clean and standardize

In [0]:
df_dengue_spatial = (
    df_raw
    # Keeps only data rows. This avoids row_number over an unordered window.
    .filter(trim(col("_c0").cast("string")).rlike(r"^[0-9]+$"))
    .select(
        to_date(trim(col("_c1").cast("string")), "dd/MM/yyyy").alias("data"),
        clean_int("_c2").alias("idade"),
        normalize_text(col("_c3")).alias("sexo"),
        normalize_text(col("_c4")).alias("bairro"),
        trim(col("_c5").cast("string")).alias("unidade_notificacao"),
        normalize_text(col("_c6")).alias("exame"),
        lit("dengue").alias("doenca")
    )
    .filter(col("bairro").isNotNull())
    .filter(col("bairro") != "")
)

display(df_dengue_spatial)
df_dengue_spatial.printSchema()

## 4. Save transformed table

In [0]:
df_dengue_spatial.write     .format("delta")     .mode("overwrite")     .option("overwriteSchema", "true")     .saveAsTable(OUTPUT_TABLE)

## 5. Validate

In [0]:
df_saved = spark.table(OUTPUT_TABLE)

print("Rows:", df_saved.count())

display(
    df_saved
    .groupBy("bairro")
    .count()
    .orderBy(col("count").desc(), "bairro")
)

display(df_saved.orderBy("data", "bairro"))